In [1]:
!git clone https://github.com/iamsyeikh/seasonal-weighted-average-imputation-for-time-series-data.git

fatal: destination path 'seasonal-weighted-average-imputation-for-time-series-data' already exists and is not an empty directory.


In [2]:
import sys
sys.path.append('seasonal-weighted-average-imputation-for-time-series-data/algorithm')

In [3]:
# Import the imputation methods

from mean_imputation import mean_imputation
from median import median_imputation
from forward_fill import forward_fill_imputation
from new import new_adaptive_weighted_seasonal_imputation
from linear_interpolation import linear_interpolation

In [4]:
import pandas as pd

In [5]:
# Load the dataset with missing values
data_missing = pd.read_excel("Challenge#1.xlsx", index_col=0, header=0, parse_dates=True)
target_column = ["CO"]

In [6]:
# Jumlah missing values pada kolom target
data_missing.isna().sum()

CO    62
dtype: int64

In [7]:
# Copy the original data for each imputation method
data_mean = data_missing.copy()
data_median = data_missing.copy()
data_ffill = data_missing.copy()
data_interpolation = data_missing.copy()
data_seasonal = data_missing.copy()

In [8]:
# Mean
for col in target_column:
    data_mean[col] = mean_imputation(data_mean[col])

# Median
for col in target_column:
    data_median[col] = median_imputation(data_median[col])

# Forward Fill
for col in target_column:
    data_ffill[col] = forward_fill_imputation(data_ffill[col])

# Interpolasi
for col in target_column:
    data_interpolation[col] = linear_interpolation(
        data_interpolation[col]
    )

# Adaptive Weighted Seasonal
for col in target_column:
    data_seasonal[col] = new_adaptive_weighted_seasonal_imputation(
        data_seasonal[col],
        seasonal_period=7
    )

In [9]:
datasets = {
    "Mean": data_mean,
    "Median": data_median,
    "Forward Fill": data_ffill,
    "Interpolation": data_interpolation,
    "Adaptive Seasonal": data_seasonal
}

for method, df in datasets.items():
    print(f"\n{method}")
    print(df[target_column].isna().sum())


Mean
CO    0
dtype: int64

Median
CO    0
dtype: int64

Forward Fill
CO    0
dtype: int64

Interpolation
CO    0
dtype: int64

Adaptive Seasonal
CO    0
dtype: int64


In [10]:
# Membuat tabel hasil
hasil = pd.DataFrame()

# Kolom non-target tetap diambil dari data asli
non_target_columns = [col for col in data_missing.columns if col not in target_column]

hasil[non_target_columns] = data_missing[non_target_columns]

# Masukkan hasil kelima metode
for col in target_column:
    hasil[f"{col}_Mean"] = data_mean[col]
    hasil[f"{col}_Median"] = data_median[col]
    hasil[f"{col}_ForwardFill"] = data_ffill[col]
    hasil[f"{col}_Interpolation"] = data_interpolation[col]
    hasil[f"{col}_Seasonal"] = data_seasonal[col]

# Tampilkan tabel
hasil

,CO_Mean,CO_Median,CO_ForwardFill,CO_Interpolation,CO_Seasonal
Date,,,,,
2016-01-01,5.750000,5.750,5.75,5.750,5.7500
2016-01-02,3.450000,3.450,3.45,3.450,3.4500
2016-01-03,3.800000,3.800,3.80,3.800,3.8000
2016-01-04,3.550000,3.550,3.55,3.550,3.5500
2016-01-05,3.650000,3.650,3.65,3.650,3.6500
...,...,...,...,...,...
2016-12-27,1.669934,1.425,2.41,2.195,2.4375
2016-12-28,1.980000,1.980,1.98,1.980,1.9800
2016-12-29,1.669934,1.425,1.98,2.160,2.8525


In [11]:
# Tentukan target
target = "CO"

# Cari baris yang CO-nya hilang pada data asli
missing_index = data_missing[data_missing[target].isna()].index

# Buat tabel khusus data yang diimputasi
hasil_imputasi = pd.DataFrame(index=missing_index)

hasil_imputasi["CO_Asli"] = data_missing.loc[missing_index, target]

hasil_imputasi["Mean"] = data_mean.loc[missing_index, target]

hasil_imputasi["Median"] = data_median.loc[missing_index, target]

hasil_imputasi["Forward_Fill"] = data_ffill.loc[missing_index, target]

hasil_imputasi["Interpolation"] = data_interpolation.loc[missing_index, target]

hasil_imputasi["Adaptive_Seasonal"] = data_seasonal.loc[missing_index, target]

# Tampilkan tabel
hasil_imputasi

,CO_Asli,Mean,Median,Forward_Fill,Interpolation,Adaptive_Seasonal
Date,,,,,,
2016-07-05,NaN,1.669934,1.425,1.70,1.555,1.265000
2016-07-07,NaN,1.669934,1.425,1.41,2.010,1.445000
2016-07-11,NaN,1.669934,1.425,1.30,1.590,1.321667
2016-07-12,NaN,1.669934,1.425,1.30,1.880,1.625000
2016-07-16,NaN,1.669934,1.425,1.26,1.925,1.470833
...,...,...,...,...,...,...
2016-12-21,NaN,1.669934,1.425,3.40,3.790,2.885000
2016-12-23,NaN,1.669934,1.425,4.18,5.205,4.064167
2016-12-25,NaN,1.669934,1.425,6.23,4.320,3.090000


In [12]:
hasil.to_csv("hasil_imputasi_5_metode.csv", index=False)